In [1]:
# Code to map Maven Lead elements to Origami Lead elements

In [2]:
from transformers import BertTokenizer, BertModel
import torch


In [3]:
import pandas as pd
import numpy as np

In [4]:
# Read maven elements
df_maven = pd.read_csv('output/maven_all_with_mmg_and_cste.csv',na_filter=False)
len(df_maven)

9501

In [5]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9501 entries, 0 to 9500
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Model                  9501 non-null   object
 1   question_number        9501 non-null   int64 
 2   depth                  9501 non-null   int64 
 3   Question               9501 non-null   object
 4   Question.Package.ID    9501 non-null   object
 5   Question.Package.Name  9501 non-null   object
 6   Message                9501 non-null   object
 7   mmg_element            9501 non-null   object
 8   cste_element           9501 non-null   object
dtypes: int64(2), object(7)
memory usage: 668.2+ KB


In [6]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If not space, digits, uppercase letters, lowercase letters, or underscore, ignore it
        if not (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=''
        else:
            new_name+=ch
    return new_name.strip()

In [7]:
# Clean up column names in both data sets
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [8]:
df_maven.loc[:,'Model'].value_counts(dropna=False)

Model
General                  3866
STD                      2425
Coinfection               923
Hepatitis                 901
Childhood Lead            606
Cluster                   425
Lead investigation        255
STD EHR and screening      85
Aggregate                  15
Name: count, dtype: int64

In [9]:
df_maven_lead = df_maven[df_maven['Model']=='Childhood Lead']
len(df_maven_lead)

606

In [10]:
# No duplicates
df_maven_lead[df_maven_lead.duplicated(['Question'],keep=False)]

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element


In [11]:
# Read origami elements
df_origami = pd.read_csv('origami/Custom__CaseLead.csv',na_filter=False)
len(df_origami)

783

In [12]:
df_origami.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 783 entries, 0 to 782
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Field                 783 non-null    object
 1   Type                  783 non-null    object
 2   Label                 783 non-null    object
 3   Required in Database  783 non-null    bool  
 4   Can't Protect         783 non-null    bool  
 5   FK Table              783 non-null    object
 6   CodeTypeID            783 non-null    object
 7   Static Codes          783 non-null    object
 8   Field Notes           783 non-null    object
 9   Global Comments       783 non-null    object
 10  Local Comments        783 non-null    object
dtypes: bool(2), object(9)
memory usage: 56.7+ KB


In [13]:
df_origami.loc[:,'Label'].value_counts(dropna=False)

Label
                                       382
HI_lead_18                               2
HI_lead_26                               2
Have they all been tested for lead?      2
HI_lead_8                                2
                                      ... 
HG_EXP_BROKEN_ITEM                       1
HG_EXP_EMISSIONS                         1
HG_EXP_PRODUCT                           1
HG_EXP_JOB_ACTIVITY                      1
HI_lead_13_details                       1
Name: count, Length: 398, dtype: int64

In [14]:
# Get the two columns of interest and non-blank values
df_origami = df_origami[['Field','Label']]
df_origami = df_origami[df_origami['Label']!='']
len(df_origami)

401

In [15]:
# Process the column values (eliminate the values in square brackets)
df_origami.loc[:,'Label'] = [ label.split(']')[1].strip() if ']' in label else label.strip() for label in df_origami.loc[:,'Label']]

In [16]:
df_origami.loc[:,'Label'].value_counts()

Label
HI_lead_26                             2
HI_lead_8                              2
HI_lead_18                             2
Investigation Completed                2
Have they all been tested for lead?    2
                                      ..
HG_FILLING_REPAIR                      1
HG_FILLING                             1
HG_WORK_SITE                           1
HG_COSMETIC                            1
HI_lead_13_details                     1
Name: count, Length: 394, dtype: int64

In [17]:
df_origami.reset_index(drop=True)

,Field,Label
0,CaseLeadID,Case Lead ID
1,ClientID,Client
2,EntryUserID,Entry User
3,EntryDate,Entry Date
4,ModifiedUserID,Modified User
...,...,...
396,CustomNarrative9,HI_lead_24
397,CustomNarrative10,HI_lead_28
398,CustomNarrative11,HI_lead_8
399,CustomNarrative12,lead_investigation_notes


In [18]:
# duplicates labels exist but they are labels to different questions, so keep them
df_origami[df_origami.duplicated(['Label'],keep=False)]

,Field,Label
67,CustomText6,Address
70,CustomText9,Name
73,CustomText12,Name
77,CustomText16,Address
100,CustomDate14,Investigation Completed
102,CustomDate16,Investigation Completed
231,CustomText93,HI_lead_8
349,CustomCode203ID,HI_lead_18
354,CustomCode208ID,HI_lead_26
355,CustomCode209ID,Have they all been tested for lead?


In [19]:
# Do a semantic match between Maven Lead elements and Origami Lead labels

In [20]:
# Create an empty co-occurrence matrix
matrix = np.zeros((len(df_maven_lead),len(df_origami)),dtype=np.float32)
matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [21]:
df_maven_lead.rename(columns={'Question':'Maven_DE'},inplace=True)

C:\Users\s.chintabathina\AppData\Local\Temp\ipykernel_88440\2962969232.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_maven_lead.rename(columns={'Question':'Maven_DE'},inplace=True)


In [22]:
df_matrix = pd.DataFrame(matrix,index=df_maven_lead.loc[:,'Maven_DE'],columns=df_origami.loc[:,'Label'])

In [23]:
df_matrix

Label,Case Lead ID,Client,Entry User,Entry Date,Modified User,Modified Date,IncidentID,"If pregnant, was blood lead testing",Is this a Child Case (<16 yrs) or Ad,Specimen Source,...,PROBABLE_LEAD_RESOURCES,HI_lead_6_no_children,HI_lead_17,HI_lead_19,HI_lead_20,HI_lead_24,HI_lead_28,HI_lead_8,lead_investigation_notes,HI_lead_13_details
Maven_DE,,,,,,,,,,,,,,,,,,,,,
ANY_OVER_5_OUTPUT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
VENOUS_CONFIRMED_DISPLAY,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MAX_VENOUS_BLL_RANGE_OUTPUT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MAX_VENOUS_BLL_OUTPUT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MAX_VENOUS_BLL_DATE_OUTPUT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DR_OVERDUE_TEST_DATE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DR_MAILING_ADDRESS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
DR_MAILING_CITY,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
df_matrix.info()

<class 'pandas.core.frame.DataFrame'>
Index: 606 entries, ANY_OVER_5_OUTPUT to DR_MAILING_ZIPCODE
Columns: 401 entries, Case Lead ID to HI_lead_13_details
dtypes: float32(401)
memory usage: 954.0+ KB


In [25]:
# Load pre-trained BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
def calculate_bert_similarity(pair):     
    tokens = tokenizer(pair, padding = "max_length",max_length=65,return_tensors='pt')
    with torch.no_grad():
        outputs = model(**tokens)
    
    # Extract embeddings for the [CLS] token
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    
    similarity = torch.nn.functional.cosine_similarity(cls_embedding[0], cls_embedding[1], dim=0)
    return similarity

In [27]:
df_matrix.index = [str.replace(maven_de,'_',' ') for maven_de in df_matrix.index]
df_matrix.columns = [str.replace(col,'_',' ') for col in df_matrix.columns]

In [28]:
# Computing max length for index
max=0
for idx in df_matrix.index:
    if len(idx)>max:
        max=len(idx)
print(max)

63


In [ ]:
# Compare BERT score for every column which are data elements from df_mmg
for col in df_matrix.columns:
    try:
        df_matrix.loc[:,col] = [calculate_bert_similarity((maven_de,col)) for maven_de in df_matrix.index]
    except:
        print('col with issue',col)
        #print(df_matrix)

col with issue Address
                             Case Lead ID    Client  Entry User  Entry Date  \
ANY OVER 5 OUTPUT                0.918876  0.901913    0.896156    0.875946   
VENOUS CONFIRMED DISPLAY         0.928475  0.964063    0.877666    0.936718   
MAX VENOUS BLL RANGE OUTPUT      0.861013  0.819373    0.831207    0.840793   
MAX VENOUS BLL OUTPUT            0.866751  0.820878    0.841372    0.838579   
MAX VENOUS BLL DATE OUTPUT       0.881023  0.831005    0.857748    0.876520   
...                                   ...       ...         ...         ...   
DR OVERDUE TEST DATE             0.925761  0.946586    0.876383    0.949674   
DR MAILING ADDRESS               0.901532  0.884471    0.891142    0.925532   
DR MAILING CITY                  0.936157  0.934034    0.898555    0.918672   
DR MAILING STATE                 0.916963  0.857669    0.918261    0.888770   
DR MAILING ZIPCODE               0.719434  0.629498    0.713665    0.645919   

                            

In [35]:
df_matrix.to_csv('output/bert_matrix_lead.csv')

In [ ]:
# Columns missing underscore so will be restoring to original names
df_matrix.columns

Index(['Case Lead ID', 'Client', 'Entry User', 'Entry Date', 'Modified User',
       'Modified Date', 'IncidentID', 'If pregnant, was blood lead testing',
       'Is this a Child Case (<16 yrs) or Ad', 'Specimen Source',
       ...
       'PROBABLE LEAD RESOURCES', 'HI lead 6 no children', 'HI lead 17',
       'HI lead 19', 'HI lead 20', 'HI lead 24', 'HI lead 28', 'HI lead 8',
       'lead investigation notes', 'HI lead 13 details'],
      dtype='object', length=401)

In [42]:
#list(df_origami.Label)

In [41]:
df_matrix.columns = list(df_origami.Label)
df_matrix.columns

Index(['Case Lead ID', 'Client', 'Entry User', 'Entry Date', 'Modified User',
       'Modified Date', 'IncidentID', 'If pregnant, was blood lead testing',
       'Is this a Child Case (<16 yrs) or Ad', 'Specimen Source',
       ...
       'PROBABLE_LEAD_RESOURCES', 'HI_lead_6_no_children', 'HI_lead_17',
       'HI_lead_19', 'HI_lead_20', 'HI_lead_24', 'HI_lead_28', 'HI_lead_8',
       'lead_investigation_notes', 'HI_lead_13_details'],
      dtype='object', length=401)

In [43]:
df_matrix.index


Index(['ANY OVER 5 OUTPUT', 'VENOUS CONFIRMED DISPLAY',
       'MAX VENOUS BLL RANGE OUTPUT', 'MAX VENOUS BLL OUTPUT',
       'MAX VENOUS BLL DATE OUTPUT', 'BLL DECLINE OUTPUT',
       'BLL TOTAL DECLINING OUTPUT', 'LATE FOLLOW UP OVER RIDE',
       'TESTING DUE DATE OUTPUT', 'LATEST VENOUS BLL OUTPUT',
       ...
       'DR PHONE NUMBER', 'DR DISCUSSION', 'DR EMAIL ADDRESS',
       'MANUAL WRITE IN FOLLOW UP', 'DR OVERDUE TEST', 'DR OVERDUE TEST DATE',
       'DR MAILING ADDRESS', 'DR MAILING CITY', 'DR MAILING STATE',
       'DR MAILING ZIPCODE'],
      dtype='object', length=606)

In [47]:
#list(df_maven_lead.loc[:,'Maven_DE'])

In [48]:
df_matrix.index =list(df_maven_lead.loc[:,'Maven_DE'])

In [49]:
df_matrix.index

Index(['ANY_OVER_5_OUTPUT', 'VENOUS_CONFIRMED_DISPLAY',
       'MAX_VENOUS_BLL_RANGE_OUTPUT', 'MAX_VENOUS_BLL_OUTPUT',
       'MAX_VENOUS_BLL_DATE_OUTPUT', 'BLL_DECLINE_OUTPUT',
       'BLL_TOTAL_DECLINING_OUTPUT', 'LATE_FOLLOW_UP_OVER_RIDE',
       'TESTING_DUE_DATE_OUTPUT', 'LATEST_VENOUS_BLL_OUTPUT',
       ...
       'DR_PHONE_NUMBER', 'DR_DISCUSSION', 'DR_EMAIL_ADDRESS',
       'MANUAL_WRITE_IN_FOLLOW_UP', 'DR_OVERDUE_TEST', 'DR_OVERDUE_TEST_DATE',
       'DR_MAILING_ADDRESS', 'DR_MAILING_CITY', 'DR_MAILING_STATE',
       'DR_MAILING_ZIPCODE'],
      dtype='object', length=606)

In [50]:
df_matrix

,Case Lead ID,Client,Entry User,Entry Date,Modified User,Modified Date,IncidentID,"If pregnant, was blood lead testing",Is this a Child Case (<16 yrs) or Ad,Specimen Source,...,PROBABLE_LEAD_RESOURCES,HI_lead_6_no_children,HI_lead_17,HI_lead_19,HI_lead_20,HI_lead_24,HI_lead_28,HI_lead_8,lead_investigation_notes,HI_lead_13_details
ANY_OVER_5_OUTPUT,0.918876,0.901913,0.896156,0.875946,0.920021,0.867591,0.854042,0.802096,0.785981,0.842165,...,0.849997,0.858877,0.877492,0.888641,0.892178,0.873345,0.869892,0.0,0.886342,0.909916
VENOUS_CONFIRMED_DISPLAY,0.928475,0.964063,0.877666,0.936718,0.944091,0.925968,0.894028,0.830589,0.744928,0.924435,...,0.819254,0.862154,0.916536,0.933154,0.930105,0.916518,0.913783,0.0,0.961800,0.940994
MAX_VENOUS_BLL_RANGE_OUTPUT,0.861013,0.819373,0.831207,0.840793,0.837677,0.848758,0.857019,0.818121,0.791092,0.800395,...,0.847472,0.830135,0.828804,0.834101,0.836891,0.839906,0.837303,0.0,0.816257,0.835668
MAX_VENOUS_BLL_OUTPUT,0.866751,0.820878,0.841372,0.838579,0.839086,0.848863,0.863522,0.837748,0.798254,0.797146,...,0.869414,0.833597,0.829691,0.835403,0.837549,0.843423,0.837779,0.0,0.821416,0.840441
MAX_VENOUS_BLL_DATE_OUTPUT,0.881023,0.831005,0.857748,0.876520,0.851581,0.868407,0.871685,0.853685,0.828187,0.831955,...,0.864436,0.857350,0.850000,0.855632,0.860163,0.863769,0.853429,0.0,0.850142,0.868338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DR_OVERDUE_TEST_DATE,0.925761,0.946586,0.876383,0.949674,0.933448,0.938672,0.913794,0.850483,0.766345,0.923162,...,0.830269,0.893817,0.919305,0.931520,0.933566,0.924901,0.920214,0.0,0.947307,0.947880
DR_MAILING_ADDRESS,0.901532,0.884471,0.891142,0.925532,0.887637,0.902516,0.907977,0.845135,0.789085,0.875142,...,0.827267,0.863012,0.868889,0.877253,0.881241,0.881629,0.879504,0.0,0.881392,0.890929
DR_MAILING_CITY,0.936157,0.934034,0.898555,0.918672,0.926181,0.906094,0.899405,0.823154,0.772315,0.892659,...,0.831616,0.882337,0.910331,0.920489,0.920735,0.914755,0.913197,0.0,0.920219,0.932456
DR_MAILING_STATE,0.916963,0.857669,0.918261,0.888770,0.886269,0.882890,0.896872,0.838063,0.822278,0.834526,...,0.867419,0.848224,0.856611,0.862226,0.864953,0.863898,0.857832,0.0,0.861696,0.885352


In [127]:
#For every maven element (index), identify origami elements (column) that is the best match
outfile = open('output/maven_lead_origami_matches.csv','w')
outfile.write('maven_element,match_list\n')
for idx in df_matrix.index:
    # Looking for columns with score higher than 0.92
    v = df_matrix.loc[idx,:]>0.92
    # Get the column and its score
    filtered=[]
    for i,col in enumerate(df_matrix.columns):
        if v.iloc[i]==True:
            filtered.append({col:df_matrix.loc[idx,col]})
    outfile.write(idx+','+str(filtered)+'\n')
outfile.close()
